<a href="https://colab.research.google.com/github/AbyssWatcher07/BigData26_B_2311532021_StanleyLeonardDarmali/blob/main/Praktikum03/BD_B_P03_2311532021_StanleyLeonardDarmali.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### K-1. Menyiapkan Lingkungan dan Memindahkan Perkakas dari Praktikum 1
Langkah ini mengimpor seluruh pustaka yang dibutuhkan dan menyiapkan koneksi ke Google Drive. Selain mendefinisikan utilitas pengukuran memori dan waktu, kita juga menyiapkan variabel lineage untuk mendokumentasikan asal-usul dan jumlah baris data yang diambil agar alur audit tetap terjaga.


In [6]:
import sys, time, os, json, sqlite3, gc
import pandas as pd, numpy as np, psutil

for nama in ["pandas", "numpy", "pyarrow", "requests", "sklearn", "sqlalchemy"]:
    try:
        mod = __import__(nama)
        print(f"{nama:11s}: {mod.__version__}")
    except ImportError:
        print(f"{nama:11s}: BELUM TERPASANG")

from google.colab import drive
drive.mount("/content/drive")

DIR_MENTAH = "/content/lapisan_mentah"
DIR_KURASI = "/content/lapisan_terkurasi"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum03"

for d in (DIR_MENTAH, DIR_KURASI, DIR_SIMPAN):
    os.makedirs(d, exist_ok=True)

proses = psutil.Process(os.getpid())
catatan = []

def rss_mb():
    return proses.memory_info().rss / 1024**2

def ukur(label, fungsi):
    m0, t0 = rss_mb(), time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 2),
                    "delta_rss_mb": round(rss_mb() - m0, 1)})
    print(f"[{label}] {detik:.2f} s")
    return hasil

lineage = []
def catat_sumber(nama, asal, jumlah_baris, keterangan=""):
    lineage.append({
        "sumber": nama, "asal": asal,
        "diambil_pada": pd.Timestamp.now("UTC").isoformat(),
        "jumlah_baris": jumlah_baris, "keterangan": keterangan,
    })
    print(f"[lineage] {nama}: {jumlah_baris:,} baris")


pandas     : 2.2.3
numpy      : 2.4.6
pyarrow    : 23.0.1
requests   : 2.32.4
sklearn    : 1.6.1
sqlalchemy : 2.1.7
Mounted at /content/drive


### K-2. Akuisisi 1 — Unduhan Berkala yang Tahan Gagal
Mengunduh data transaksi taksi (Parquet) dan tabel zona (CSV) dari URL. Proses ini menggunakan mekanisme penulisan atomik (menulis ke .part lalu di-replace) dan exponential backoff agar file tidak korup jika internet terputus dan otomatis melakukan percobaan ulang secara aman.

In [7]:
import requests
import os
import time

def unduh_aman(url, tujuan, percobaan=3, jeda_awal=2):
    """Unduh dengan retry + penulisan atomik. Melewati unduhan bila file sudah ada."""
    if os.path.exists(tujuan) and os.path.getsize(tujuan) > 0:
        print("cache ditemukan:", os.path.basename(tujuan))
        return tujuan
    sementara = tujuan + ".part"
    for i in range(percobaan):
        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(sementara, "wb") as f:
                    # FIX: Use iter_content instead of raw shutil copy to auto-decompress gzip
                    for chunk in r.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
            if os.path.getsize(sementara) == 0:
                raise IOError("berkas kosong")
            os.replace(sementara, tujuan) # atomik
            return tujuan
        except Exception as e:
            jeda = jeda_awal * (2 ** i) # exponential backoff
            print(f"percobaan {i+1} gagal ({e}); menunggu {jeda}s")
            time.sleep(jeda)
    raise RuntimeError(f"gagal mengunduh setelah {percobaan} percobaan: {url}")

URL_TRIP = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
URL_ZONA = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
PATH_TRIP = os.path.join(DIR_MENTAH, "yellow_tripdata_2023-01.parquet")
PATH_ZONA = os.path.join(DIR_MENTAH, "taxi_zone_lookup.csv")

ukur("unduh trip", lambda: unduh_aman(URL_TRIP, PATH_TRIP))
ukur("unduh zona", lambda: unduh_aman(URL_ZONA, PATH_ZONA))

for pth in (PATH_TRIP, PATH_ZONA):
    print(os.path.basename(pth), "->", round(os.path.getsize(pth)/1024**2, 2), "MB")

KOLOM = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
         "trip_distance", "PULocationID", "DOLocationID", "payment_type",
         "fare_amount", "tip_amount", "total_amount"]
trip = ukur("baca trip", lambda: pd.read_parquet(PATH_TRIP, columns=KOLOM))
zona = pd.read_csv(PATH_ZONA)

catat_sumber("trip", URL_TRIP, len(trip), "Parquet bulanan TLC")
catat_sumber("zona", URL_ZONA, len(zona), "tabel dimensi 265 zona")
zona.head()

[unduh trip] 2.10 s
[unduh zona] 0.05 s
yellow_tripdata_2023-01.parquet -> 45.46 MB
taxi_zone_lookup.csv -> 0.01 MB
[baca trip] 1.33 s
[lineage] trip: 3,066,766 baris
[lineage] zona: 265 baris


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


### K-3. Akuisisi 2 — Memanggil API JSON
Mengambil data cuaca per jam secara langsung dari Open-Meteo API. Kode diatur agar menangani batas laju permintaan (rate limit) secara otomatis dengan melakukan retry berjeda khusus untuk galat HTTP yang bersifat sementara (seperti 429 atau 5xx).

In [8]:
API_CUACA = "https://archive-api.open-meteo.com/v1/archive"

def ambil_cuaca(mulai, selesai, lat=40.7128, lon=-74.0060, percobaan=3):
    parameter = {
        "latitude": lat, "longitude": lon,
        "start_date": mulai, "end_date": selesai,
        "hourly": "temperature_2m,precipitation",
        "timezone": "America/New_York",
    }
    for i in range(percobaan):
        r = requests.get(API_CUACA, params=parameter, timeout=60)
        if r.status_code == 200:
            data = r.json()
            if "hourly" not in data:
                raise ValueError("kunci 'hourly' tidak ada pada respons")
            return pd.DataFrame(data["hourly"])
        if r.status_code in (429, 500, 502, 503):
            time.sleep(2 ** i)
            continue
        r.raise_for_status()
    raise RuntimeError("API cuaca tidak merespons dengan benar")

cuaca = ukur("ambil cuaca", lambda: ambil_cuaca("2023-01-01", "2023-01-31"))
cuaca["time"] = pd.to_datetime(cuaca["time"])
cuaca = cuaca.rename(columns={"time": "jam_mulai",
                              "temperature_2m": "suhu_c",
                              "precipitation": "hujan_mm"})

catat_sumber("cuaca", API_CUACA, len(cuaca), "Open-Meteo hourly archive")
cuaca.head()


[ambil cuaca] 0.09 s
[lineage] cuaca: 744 baris


,jam_mulai,suhu_c,hujan_mm
0,2023-01-01 00:00:00,10.6,1.0
1,2023-01-01 01:00:00,10.6,0.1
2,2023-01-01 02:00:00,10.5,0.0
3,2023-01-01 03:00:00,9.8,0.0
4,2023-01-01 04:00:00,9.2,0.0


### K-4. Jalur Cadangan bila Internet Diblokir (Fungsi Sintetis Cuaca)
Fungsi alternatif untuk menghasilkan data cuaca tiruan menggunakan fungsi distribusi statistik (Normal dan Gamma) dengan seed tetap. Blok ini hanya dijalankan sebagai fallback apabila API cuaca pada K-3 benar-benar gagal merespons.

In [9]:
def cuaca_sintetis(mulai="2023-01-01", selesai="2023-01-31", seed=7):
    jam = pd.date_range(mulai, pd.Timestamp(selesai) + pd.Timedelta("23h"), freq="h")
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        "jam_mulai": jam,
        "suhu_c": np.round(rng.normal(3, 5, len(jam)), 1),
        "hujan_mm": np.round(rng.gamma(0.4, 0.8, len(jam)), 2),
    })

# Pakai hanya jika K-3 gagal:
# cuaca = cuaca_sintetis()
# catat_sumber("cuaca", "sintetis", len(cuaca), "PENGGANTI - bukan data nyata")



### K-5. Akuisisi 3 — Basis Data SQLite dengan Pemuatan Bertahap
Mensimulasikan pengambilan tabel dimensi dan sebagian data transaksi dari basis data operasional internal. Untuk mencegah kehabisan memori (Out of Memory), data transaksi yang besar ditarik menggunakan parameter chunksize sehingga diproses bagian demi bagian.

In [10]:
from sqlalchemy import create_engine

PATH_DB = os.path.join(DIR_MENTAH, "operasional.db")
engine = create_engine(f"sqlite:///{PATH_DB}")

tarif_referensi = pd.DataFrame({
    "payment_type": [1, 2, 3, 4, 5, 6],
    "nama_pembayaran": ["Kartu kredit", "Tunai", "Gratis",
                        "Sengketa", "Tidak diketahui", "Perjalanan batal"],
    "kena_biaya_admin": [1, 0, 0, 0, 0, 0],
})
tarif_referensi.to_sql("tarif_referensi", engine, if_exists="replace", index=False)

# tabel transaksi tiruan untuk latihan pembacaan bertahap
trip.head(300_000).to_sql(
    "trip_operasional", engine, if_exists="replace",
    index=False, chunksize=50_000)

print("Tabel dibuat:", pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", engine)["name"].tolist())

SQL = """
SELECT payment_type, COUNT(*) AS jumlah, AVG(total_amount) AS rata_total
FROM trip_operasional
WHERE total_amount > 0
GROUP BY payment_type
ORDER BY jumlah DESC
"""
ringkas_db = pd.read_sql(SQL, engine)
print("\n", ringkas_db)

total_baris = 0
for bagian in pd.read_sql("SELECT trip_distance FROM trip_operasional",
                          engine, chunksize=50_000):
    total_baris += len(bagian)
print("\ndibaca bertahap:", f"{total_baris:,}", "baris")

tarif_referensi = pd.read_sql("SELECT * FROM tarif_referensi", engine)
catat_sumber("tarif_referensi", PATH_DB, len(tarif_referensi),
             "tabel dimensi dari basis data operasional")


Tabel dibuat: ['tarif_referensi', 'trip_operasional']

    payment_type  jumlah  rata_total
0             1  226967   31.339172
1             2   66504   25.218836
2             4    2082   26.611114
3             3    1553   22.089691

dibaca bertahap: 300,000 baris
[lineage] tarif_referensi: 6 baris


### K-6. Profiling Kualitas Data pada Enam Dimensi
Membangun laporan kualitas data otomatis untuk mengukur anomali. Evaluasi dilakukan pada enam dimensi kualitas (kelengkapan, keunikan, validitas, konsistensi, ketepatan waktu) guna memetakan persentase data kotor yang harus ditangani pada langkah selanjutnya.

In [11]:
trip["durasi_menit"] = (
    (trip["tpep_dropoff_datetime"] - trip["tpep_pickup_datetime"])
    .dt.total_seconds() / 60)

def profil_kolom(df):
    baris = []
    for kol in df.columns:
        s = df[kol]
        baris.append({
            "kolom": kol, "tipe": str(s.dtype),
            "persen_hilang": round(100 * s.isna().mean(), 3),
            "nilai_unik": s.nunique(dropna=True),
            "contoh": s.dropna().iloc[0] if s.notna().any() else None,
        })
    return pd.DataFrame(baris)

profil = profil_kolom(trip)
display(profil)

AWAL, AKHIR = pd.Timestamp("2023-01-01"), pd.Timestamp("2023-02-01")
zona_sah = set(zona["LocationID"])

aturan = {
    "kelengkapan: passenger_count ada": trip["passenger_count"].notna(),
    "keunikan: baris tidak duplikat": ~trip.duplicated(),
    "validitas: jarak 0-100 mil": trip["trip_distance"].between(0.01, 100),
    "validitas: total_amount > 0": trip["total_amount"] > 0,
    "validitas: PULocationID dikenal": trip["PULocationID"].isin(zona_sah),
    "konsistensi: durasi 1-180 menit": trip["durasi_menit"].between(1, 180),
    "konsistensi: total >= fare": trip["total_amount"] >= trip["fare_amount"],
    "ketepatan waktu dalam Jan 2023": trip["tpep_pickup_datetime"].between(AWAL, AKHIR),
}

laporan = pd.DataFrame([
    {"aturan": nama, "lulus": int(mask.sum()), "gagal": int((~mask).sum()),
     "persen_gagal": round(100 * (~mask).mean(), 3)}
    for nama, mask in aturan.items()
]).sort_values("persen_gagal", ascending=False)

display(laporan)


,kolom,tipe,persen_hilang,nilai_unik,contoh
0,tpep_pickup_datetime,datetime64[us],0.000,1610975,2023-01-01 00:32:10
1,tpep_dropoff_datetime,datetime64[us],0.000,1611319,2023-01-01 00:40:36
2,passenger_count,float64,2.339,10,1.0
3,trip_distance,float64,0.000,4387,0.97
4,PULocationID,int64,0.000,257,161
5,DOLocationID,int64,0.000,261,141
6,payment_type,int64,0.000,5,2
7,fare_amount,float64,0.000,6873,9.3
8,tip_amount,float64,0.000,4036,0.0
9,total_amount,float64,0.000,15871,14.3


,aturan,lulus,gagal,persen_gagal
0,kelengkapan: passenger_count ada,2995023,71743,2.339
2,validitas: jarak 0-100 mil,3020816,45950,1.498
5,konsistensi: durasi 1-180 menit,3030383,36383,1.186
3,validitas: total_amount > 0,3040994,25772,0.840
6,konsistensi: total >= fare,3041743,25023,0.816
7,ketepatan waktu dalam Jan 2023,3066718,48,0.002
1,keunikan: baris tidak duplikat,3066766,0,0.000
4,validitas: PULocationID dikenal,3066766,0,0.000


### K-7. Nilai Hilang dan Duplikat
Keputusan Pra-pemrosesan: Missing value pada passenger_count ditangani menggunakan median per kelompok (jam) untuk mempertahankan distribusi lokal, didukung flag penanda _hilang. Untuk duplikasi, baris yang dihapus hanya yang terduplikasi secara penuh berdasarkan kombinasi kunci transaksi untuk mencegah hilangnya data perjalanan yang sah.

In [12]:
KOL = "passenger_count"
asli = trip[KOL]
print("persen hilang:", round(100 * asli.isna().mean(), 3))

trip["jam"] = trip["tpep_pickup_datetime"].dt.hour

strategi = {
    "1_dibuang": asli.dropna(),
    "2_isi_median": asli.fillna(asli.median()),
    "3_isi_modus": asli.fillna(asli.mode().iloc[0]),
    "4_median_perjam": asli.fillna(trip.groupby("jam")[KOL].transform("median")),
}

banding = pd.DataFrame([
    {"strategi": nama, "n": len(s), "mean": round(s.mean(), 4),
     "median": s.median(), "std": round(s.std(), 4)}
    for nama, s in strategi.items()
])
display(banding)

# Strategi yang dipilih: tandai + isi per kelompok
trip[KOL + "_hilang"] = trip[KOL].isna().astype("int8")
trip[KOL] = trip[KOL].fillna(trip.groupby("jam")[KOL].transform("median"))
trip[KOL] = trip[KOL].fillna(trip[KOL].median()) # jaring pengaman

dup_penuh = trip.duplicated().sum()
KUNCI = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
         "PULocationID", "DOLocationID", "total_amount"]
dup_kunci = trip.duplicated(subset=KUNCI).sum()

print(f"duplikat penuh: {dup_penuh:,} | duplikat pada kunci: {dup_kunci:,}")

sebelum = len(trip)
trip = trip.drop_duplicates(subset=KUNCI, keep="first").reset_index(drop=True)
print(f"{sebelum:,} -> {len(trip):,} baris")


persen hilang: 2.339


,strategi,n,mean,median,std
0,1_dibuang,2995023,1.3625,1.0,0.8961
1,2_isi_median,3066766,1.3541,1.0,0.8873
2,3_isi_modus,3066766,1.3541,1.0,0.8873
3,4_median_perjam,3066766,1.3541,1.0,0.8873


duplikat penuh: 0 | duplikat pada kunci: 1
3,066,766 -> 3,066,765 baris


### K-8. Outlier, Join dengan Tabel Referensi, dan Penggabungan Berbasis Waktu
Keputusan Pra-pemrosesan: Outlier pada harga (total_amount) hanya ditandai dan tidak dibuang, karena perjalanan ekstrem bisa mencerminkan realitas bisnis bandara. Namun, jarak yang tidak masuk akal (trip_distance) dibatasi (winsorizing) pada persentil 99.5. Penggabungan tabel zona dan cuaca diamankan secara ketat menggunakan parameter validate="many_to_one" guna mencegah ledakan baris tersembunyi.

In [13]:
def batas_iqr(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

def ringkas_outlier(df, kolom):
    hasil = []
    for kol in kolom:
        s = df[kol].dropna()
        lo, hi = batas_iqr(s)
        z = (s - s.mean()) / s.std()
        hasil.append({
            "kolom": kol, "batas_bawah": round(lo, 2), "batas_atas": round(hi, 2),
            "outlier_iqr": int(((s < lo) | (s > hi)).sum()),
            "outlier_z3": int((z.abs() > 3).sum()), "p99": round(s.quantile(0.99), 2),
        })
    return pd.DataFrame(hasil)

display(ringkas_outlier(trip, ["trip_distance", "durasi_menit", "total_amount"]))

layak = (
    (trip["trip_distance"].between(0.01, 100))
    & (trip["durasi_menit"].between(1, 180))
    & (trip["total_amount"] > 0)
)
bersih = trip.loc[layak].copy()

# tandai tarif ekstrem
lo, hi = batas_iqr(bersih["total_amount"])
bersih["tarif_ekstrem"] = (bersih["total_amount"] > hi).astype("int8")

# winsorizing
batas_atas = bersih["trip_distance"].quantile(0.995)
bersih["trip_distance_capped"] = bersih["trip_distance"].clip(upper=batas_atas)

print(f"{len(trip):,} -> {len(bersih):,} baris "
      f"({100*(1-len(bersih)/len(trip)):.2f}% dibuang)")

print("\nkunci zona unik?", zona["LocationID"].is_unique)

n0 = len(bersih)
bersih = bersih.merge(
    zona[["LocationID", "Borough", "Zone"]]
    .rename(columns={"Borough": "borough_naik", "Zone": "zona_naik"}),
    left_on="PULocationID", right_on="LocationID",
    how="left", validate="many_to_one"
)
bersih = bersih.drop(columns="LocationID")
print(f"Join 1: baris {n0:,} -> {len(bersih):,} (harus sama)")
print("zona tak dikenal:", bersih["zona_naik"].isna().sum())

# Join 2
bersih = bersih.merge(tarif_referensi[["payment_type", "nama_pembayaran"]],
                      on="payment_type", how="left", validate="many_to_one")

# Join 3: cuaca
bersih["jam_mulai"] = bersih["tpep_pickup_datetime"].dt.floor("h")
n0 = len(bersih)
bersih = bersih.merge(cuaca, on="jam_mulai", how="left", validate="many_to_one")
print(f"Join 3: baris {n0:,} -> {len(bersih):,}")
print("tanpa data cuaca:", bersih["suhu_c"].isna().sum())


,kolom,batas_bawah,batas_atas,outlier_iqr,outlier_z3,p99
0,trip_distance,-2.35,6.74,390244,67,20.06
1,durasi_menit,-9.66,35.08,170615,3175,57.25
2,total_amount,-4.55,48.65,371616,87248,101.94


3,066,765 -> 2,986,910 baris (2.60% dibuang)

kunci zona unik? True
Join 1: baris 2,986,910 -> 2,986,910 (harus sama)
zona tak dikenal: 37609
Join 3: baris 2,986,910 -> 2,986,910
tanpa data cuaca: 37


### K-9. Rekayasa Fitur: Encoding dan Scaling Tanpa Kebocoran
Keputusan Pra-pemrosesan: Melakukan standardisasi numerik (StandardScaler) dan transformasi kategorikal (OneHotEncoder). Parameter scaler dan encoder sengaja di-fit HANYA pada data latih, lalu di-transform ke data uji, guna menghindari kebocoran data (data leakage) ke dalam evaluasi model.

In [14]:
bersih["hari_minggu"] = bersih["tpep_pickup_datetime"].dt.dayofweek
bersih["akhir_pekan"] = (bersih["hari_minggu"] >= 5).astype("int8")
bersih["kecepatan_mph"] = (bersih["trip_distance"] / (bersih["durasi_menit"] / 60)).round(2)
bersih["tarif_per_mil"] = (bersih["total_amount"] / bersih["trip_distance"]).round(3)
bersih["hujan"] = (bersih["hujan_mm"].fillna(0) > 0.1).astype("int8")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

FITUR_NUM = ["trip_distance_capped", "durasi_menit", "suhu_c"]
FITUR_KAT = ["nama_pembayaran", "borough_naik"]

contoh = bersih.dropna(subset=FITUR_NUM + FITUR_KAT).sample(
    n=min(200_000, len(bersih)), random_state=42)

latih, uji = train_test_split(contoh, test_size=0.2, random_state=42)

skala = StandardScaler().fit(latih[FITUR_NUM])
latih_num = skala.transform(latih[FITUR_NUM])
uji_num = skala.transform(uji[FITUR_NUM])

print("mean latih (harus ~0):", latih_num.mean(axis=0).round(3))
print("mean uji (tidak harus 0):", uji_num.mean(axis=0).round(3))

enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
enc.fit(latih[FITUR_KAT])
latih_kat = enc.transform(latih[FITUR_KAT])

print("jumlah kolom hasil one-hot:", latih_kat.shape[1])
print("nama kolom:", enc.get_feature_names_out()[:6], "...")


mean latih (harus ~0): [-0.  0. -0.]
mean uji (tidak harus 0): [ 0.001  0.001 -0.008]
jumlah kolom hasil one-hot: 11
nama kolom: ['nama_pembayaran_Gratis' 'nama_pembayaran_Kartu kredit'
 'nama_pembayaran_Sengketa' 'nama_pembayaran_Tunai' 'borough_naik_Bronx'
 'borough_naik_Brooklyn'] ...


### K-10. Membungkus Jadi Pipeline, Memvalidasi, dan Menyimpan Berpartisi
Menggabungkan seluruh proses sebelumnya ke dalam satu fungsi terpusat (pipeline) yang bersifat idempoten (aman dijalankan berulang). Setelah lulus uji validasi skema (kontrak), dataset bersih diekspor menjadi format Parquet yang dipartisi berdasarkan borough_naik untuk mempercepat query partition pruning di masa depan.

In [15]:
import os
import shutil

KONTRAK = {
    "tpep_pickup_datetime": "datetime64[ns]",
    "trip_distance": "float64",
    "durasi_menit": "float64",
    "total_amount": "float64",
    "borough_naik": "object",
    "nama_pembayaran": "object",
}

def validasi(df, kontrak=KONTRAK):
    masalah = []
    for kol, tipe in kontrak.items():
        if kol not in df.columns:
            masalah.append(f"kolom hilang: {kol}")
        elif not str(df[kol].dtype).startswith(tipe.split("[")[0]):
            masalah.append(f"tipe {kol}: {df[kol].dtype} != {tipe}")

    if df.empty:
        masalah.append("DataFrame kosong")
    if len(df) != len(df.drop_duplicates(subset=KUNCI)):
        masalah.append("masih ada duplikat pada kunci")

    if masalah:
        raise AssertionError("Validasi gagal:\n- " + "\n- ".join(masalah))
    print(f"Validasi lulus: {len(df):,} baris x {df.shape[1]} kolom")
    return df

validasi(bersih)

def pipeline(path_trip, path_zona, df_cuaca, df_tarif):
    """Satu fungsi, idempoten: input mentah -> DataFrame tervalidasi."""
    df = pd.read_parquet(path_trip, columns=KOLOM)
    z = pd.read_csv(path_zona)

    df["durasi_menit"] = ((df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"])
                          .dt.total_seconds() / 60)
    df["jam"] = df["tpep_pickup_datetime"].dt.hour
    df["passenger_count"] = df["passenger_count"].fillna(
        df.groupby("jam")["passenger_count"].transform("median"))

    df = df.drop_duplicates(subset=KUNCI)
    df = df[df["trip_distance"].between(0.01, 100)
            & df["durasi_menit"].between(1, 180)
            & (df["total_amount"] > 0)]

    df = (df.merge(z[["LocationID", "Borough"]]
                 .rename(columns={"Borough": "borough_naik"}),
                 left_on="PULocationID", right_on="LocationID",
                 how="left", validate="many_to_one")
          .drop(columns="LocationID")
          .merge(df_tarif[["payment_type", "nama_pembayaran"]],
                 on="payment_type", how="left", validate="many_to_one"))

    df["jam_mulai"] = df["tpep_pickup_datetime"].dt.floor("h")
    df = df.merge(df_cuaca, on="jam_mulai", how="left", validate="many_to_one")
    return validasi(df)

hasil = ukur("pipeline penuh", lambda: pipeline(PATH_TRIP, PATH_ZONA, cuaca, tarif_referensi))

OUT = os.path.join(DIR_KURASI, "trips_bersih")
hasil["tanggal"] = hasil["tpep_pickup_datetime"].dt.date.astype(str)

ukur("tulis parquet berpartisi", lambda: hasil.to_parquet(
    OUT, partition_cols=["borough_naik"], index=False))

print("partisi yang terbentuk:", sorted(os.listdir(OUT))[:5], "...")

# Bukti idempotensi: jalankan ulang, hasil harus identik
ulang = pipeline(PATH_TRIP, PATH_ZONA, cuaca, tarif_referensi)
print("idempoten?", len(ulang) == len(hasil))

# Artefak yang dikumpulkan
laporan.to_csv(f"{DIR_SIMPAN}/laporan_kualitas.csv", index=False)
pd.DataFrame(lineage).to_csv(f"{DIR_SIMPAN}/lineage.csv", index=False)
pd.DataFrame(catatan).to_csv(f"{DIR_SIMPAN}/pengukuran_kinerja.csv", index=False)
shutil.make_archive(f"{DIR_SIMPAN}/trips_bersih", "zip", OUT)

Validasi lulus: 2,986,910 baris x 26 kolom
Validasi lulus: 2,986,910 baris x 17 kolom
[pipeline penuh] 7.12 s
[tulis parquet berpartisi] 3.37 s
partisi yang terbentuk: ['borough_naik=Bronx', 'borough_naik=Brooklyn', 'borough_naik=EWR', 'borough_naik=Manhattan', 'borough_naik=Queens'] ...
Validasi lulus: 2,986,910 baris x 17 kolom
idempoten? True


'/content/drive/MyDrive/BigData/Praktikum03/trips_bersih.zip'

### K-11. Pra-pemrosesan Setara dengan PySpark (Tambahan)
Menjalankan replika pipeline pembersihan pada skala terdistribusi menggunakan PySpark. Tabel referensi zona yang berukuran kecil dipaksa menggunakan F.broadcast() saat di-join agar mesin tidak perlu melakukan shuffle yang berat, sehingga komputasi berjalan jauh lebih cepat.

In [16]:
!pip install -q pyspark==3.5.1
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("BD-P03").master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

sdf = spark.read.parquet(PATH_TRIP).select(*KOLOM)
szona = spark.createDataFrame(zona[["LocationID", "Borough"]])

sbersih = (sdf
           # Fix: Menggunakan unix_timestamp alih-alih cast("long")
           .withColumn("durasi_menit",
                       (F.unix_timestamp("tpep_dropoff_datetime")
                        - F.unix_timestamp("tpep_pickup_datetime")) / 60)
           .filter(F.col("trip_distance").between(0.01, 100))
           .filter(F.col("durasi_menit").between(1, 180))
           .filter(F.col("total_amount") > 0)
           .dropDuplicates(["tpep_pickup_datetime", "tpep_dropoff_datetime",
                            "PULocationID", "DOLocationID", "total_amount"])
           .join(F.broadcast(szona),
                 sdf.PULocationID == szona.LocationID, "left")
           .drop("LocationID"))

ukur("spark: hitung baris bersih", lambda: sbersih.count())
sbersih.groupBy("Borough").count().orderBy(F.desc("count")).show(10, False)

sbersih.write.mode("overwrite").partitionBy("Borough") \
    .parquet("/content/lapisan_terkurasi/trips_spark")

sbersih.explain()
spark.stop()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
[spark: hitung baris bersih] 14.97 s
+-------------+-------+
|Borough      |count  |
+-------------+-------+
|Manhattan    |2659609|
|Queens       |270315 |
|Unknown      |37609  |
|Brooklyn     |15724  |
|Bronx        |3074   |
|NaN          |340    |
|Staten Island|211    |
|EWR          |28     |
+-------------+-------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#245, trip_distance#247, PULocationID#7L, DOLocationID#8L, payment_type#249L, fare_amount#251, tip_amount#253, total_am

### P. Studi Kasus: Tim Harga Dinamis (Pricing)
Menyiapkan tabel analitik khusus yang meringkas agregasi jumlah perjalanan, rata-rata tarif, kecepatan, dan curah hujan per jam untuk setiap wilayah (borough). Pengelompokan ini mengabaikan sel-sel dengan jumlah sampel kurang dari 30 agar kesimpulan analitis terkait cuaca tidak bias akibat data yang terlalu sedikit.

In [17]:
tabel_analitik = (bersih
    .assign(jam_mulai = bersih["tpep_pickup_datetime"].dt.floor("h"))
    .groupby(["borough_naik", "jam_mulai"], observed=True)
    .agg(jumlah_perjalanan=("total_amount", "size"),
         rata_tarif_per_mil=("tarif_per_mil", "median"),
         rata_kecepatan=("kecepatan_mph", "median"),
         suhu_c=("suhu_c", "first"), hujan=("hujan", "max"))
    .reset_index())

tabel_analitik = tabel_analitik[tabel_analitik["jumlah_perjalanan"] >= 30]

print(tabel_analitik
      .groupby(["borough_naik", "hujan"], observed=True)["rata_tarif_per_mil"]
      .median().unstack())


hujan              0         1
borough_naik                  
Brooklyn       6.943   7.65500
Manhattan     10.500  11.42100
Queens         5.451   5.65175
Unknown        8.724   9.82550


### Q. Latihan Mandiri (Q1 - Q5)


In [18]:
# --- Latihan 1: Modifikasi unduh_aman dengan cetak kecepatan MB/s ---
def unduh_aman_cepat(url, tujuan, percobaan=3, jeda_awal=2):
    if os.path.exists(tujuan) and os.path.getsize(tujuan) > 0:
        print("cache ditemukan:", os.path.basename(tujuan))
        return tujuan
    sementara = tujuan + ".part"
    for i in range(percobaan):
        try:
            start_time = time.perf_counter()
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(sementara, "wb") as f:
                    shutil.copyfileobj(r.raw, f)
            size_mb = os.path.getsize(sementara) / (1024 * 1024)
            duration = time.perf_counter() - start_time
            if size_mb == 0:
                raise IOError("berkas kosong")
            os.replace(sementara, tujuan)
            print(f"Unduhan selesai: {size_mb:.2f} MB dalam {duration:.2f} s ({size_mb/duration:.2f} MB/s)")
            return tujuan
        except Exception as e:
            jeda = jeda_awal * (2 ** i)
            print(f"percobaan {i+1} gagal ({e}); menunggu {jeda}s")
            time.sleep(jeda)
    raise RuntimeError(f"gagal mengunduh setelah {percobaan} percobaan: {url}")

print("\n[Latihan 1]")
TEST_PATH = os.path.join(DIR_MENTAH, "test_download.csv")
# Jalankan 2x untuk lihat efek cache
unduh_aman_cepat(URL_ZONA, TEST_PATH)
unduh_aman_cepat(URL_ZONA, TEST_PATH)

# --- Latihan 2: Tambah Aturan di K-6 ---
print("\n[Latihan 2]")
aturan_baru = {
    "validitas: tip_amount >= 0": trip["tip_amount"] >= 0,
    "konsistensi: fare + tip <= total": (trip["fare_amount"] + trip["tip_amount"]) <= trip["total_amount"]
}
laporan_baru = pd.DataFrame([
    {"aturan": nama, "lulus": int(mask.sum()), "gagal": int((~mask).sum()), "persen_gagal": round(100 * (~mask).mean(), 3)}
    for nama, mask in aturan_baru.items()
])
display(laporan_baru)

# --- Latihan 3: Join DOLocationID ---
print("\n[Latihan 3]")
bersih_q3 = bersih.merge(
    zona[["LocationID", "Borough"]].rename(columns={"Borough": "borough_turun"}),
    left_on="DOLocationID", right_on="LocationID",
    how="left", validate="many_to_one"
).drop(columns="LocationID")

top_10_rute = bersih_q3.groupby(["borough_naik", "borough_turun"]).size().sort_values(ascending=False).head(10)
print(top_10_rute)

# --- Latihan 4: MinMaxScaler vs StandardScaler ---
print("\n[Latihan 4]")
from sklearn.preprocessing import MinMaxScaler
skala_minmax = MinMaxScaler().fit(latih[FITUR_NUM])
latih_minmax = skala_minmax.transform(latih[FITUR_NUM])
print("StandardScaler (mean, std):", latih_num.mean(axis=0).round(2), latih_num.std(axis=0).round(2))
print("MinMaxScaler (min, max):", latih_minmax.min(axis=0).round(2), latih_minmax.max(axis=0).round(2))

# --- Latihan 5: Bukti validate='many_to_one' ---
print("\n[Latihan 5]")
zona_ganda = pd.concat([zona, zona.iloc[[0]]]) # duplicate first row intentionally
try:
    _ = bersih.merge(zona_ganda[["LocationID", "Borough"]],
                     left_on="PULocationID", right_on="LocationID",
                     how="left", validate="many_to_one")
except Exception as e:
    print(f"Error tertangkap seperti yang diharapkan:\n{type(e).__name__}: {e}")

# --- Latihan 6: Pipeline Inkremental & Uji Idempotensi ---
print("\n[Latihan 6]")
LOG_PROSES = os.path.join(DIR_KURASI, "processed_months.txt")
def baca_log():
    if os.path.exists(LOG_PROSES):
        with open(LOG_PROSES, "r") as f:
            return set(f.read().splitlines())
    return set()

def catat_log(bulan_str):
    with open(LOG_PROSES, "a") as f:
        f.write(bulan_str + "\n")

def pipeline_inkremental(bulan_str, url_trip):
    sudah_diproses = baca_log()
    if bulan_str in sudah_diproses:
        print(f"Bulan {bulan_str} sudah ada di direktori keluaran. Dilewati (Idempoten).")
        return None

    print(f"Memproses bulan {bulan_str}...")
    path_trip_baru = os.path.join(DIR_MENTAH, f"yellow_tripdata_{bulan_str}.parquet")
    unduh_aman_cepat(url_trip, path_trip_baru)

    hasil_baru = pipeline(path_trip_baru, PATH_ZONA, cuaca, tarif_referensi)

    OUT_KURASI = os.path.join(DIR_KURASI, "trips_bersih")
    hasil_baru["tanggal"] = hasil_baru["tpep_pickup_datetime"].dt.date.astype(str)
    # Menulis secara berpartisi; PyArrow akan menambahkan file ke folder partisi secara otomatis
    hasil_baru.to_parquet(OUT_KURASI, partition_cols=["borough_naik"], index=False)

    catat_log(bulan_str)
    print(f"Selesai! {len(hasil_baru):,} baris ditambahkan ke {OUT_KURASI}.")
    return hasil_baru
URL_JAN = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
print("Eksekusi Pertama:")
pipeline_inkremental("2023-01", URL_JAN)
print("\nEksekusi Kedua (Harus dilewati):")
pipeline_inkremental("2023-01", URL_JAN)



[Latihan 1]
Unduhan selesai: 0.00 MB dalam 0.10 s (0.03 MB/s)
cache ditemukan: test_download.csv

[Latihan 2]


,aturan,lulus,gagal,persen_gagal
0,validitas: tip_amount >= 0,3066540,225,0.007
1,konsistensi: fare + tip <= total,3041605,25160,0.820



[Latihan 3]
borough_naik  borough_turun
Manhattan     Manhattan        2491566
Queens        Manhattan         155261
Manhattan     Queens             82354
              Brooklyn           62986
Queens        Queens             59146
              Brooklyn           42187
Unknown       Manhattan          18432
              Unknown            13562
Manhattan     Bronx               8865
Brooklyn      Brooklyn            7910
dtype: int64

[Latihan 4]
StandardScaler (mean, std): [-0.  0. -0.] [1. 1. 1.]
MinMaxScaler (min, max): [0. 0. 0.] [1. 1. 1.]

[Latihan 5]
Error tertangkap seperti yang diharapkan:
MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

[Latihan 6]
Eksekusi Pertama:
Memproses bulan 2023-01...
cache ditemukan: yellow_tripdata_2023-01.parquet
Validasi lulus: 2,986,910 baris x 17 kolom
Selesai! 2,986,910 baris ditambahkan ke /content/lapisan_terkurasi/trips_bersih.

Eksekusi Kedua (Harus dilewati):
Bulan 2023-01 sudah ada di direktori keluar

R.1 & R.2 (Pipeline Dua Bulan & Laporan Kualitas Komparatif)

In [19]:
print("\n[Tugas R.1 & R.2: Pipeline Dua Bulan & Profil Komparatif]")
URL_JUL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-07.parquet"
PATH_JUL = os.path.join(DIR_MENTAH, "yellow_tripdata_2023-07.parquet")

unduh_aman_cepat(URL_JUL, PATH_JUL)
trip_jul = pd.read_parquet(PATH_JUL, columns=KOLOM)
trip_jul["durasi_menit"] = ((trip_jul["tpep_dropoff_datetime"] - trip_jul["tpep_pickup_datetime"]).dt.total_seconds() / 60)

AWAL_JUL, AKHIR_JUL = pd.Timestamp("2023-07-01"), pd.Timestamp("2023-08-01")
aturan_jul = {
    "kelengkapan: passenger_count ada": trip_jul["passenger_count"].notna(),
    "keunikan: baris tidak duplikat": ~trip_jul.duplicated(),
    "validitas: jarak 0-100 mil": trip_jul["trip_distance"].between(0.01, 100),
    "validitas: total_amount > 0": trip_jul["total_amount"] > 0,
    "validitas: PULocationID dikenal": trip_jul["PULocationID"].isin(zona_sah),
    "konsistensi: durasi 1-180 menit": trip_jul["durasi_menit"].between(1, 180),
    "konsistensi: total >= fare": trip_jul["total_amount"] >= trip_jul["fare_amount"],
    "ketepatan waktu dalam target bulan": trip_jul["tpep_pickup_datetime"].between(AWAL_JUL, AKHIR_JUL),
}

laporan_jul = pd.DataFrame([
    {"aturan": nama, "persen_gagal_juli": round(100 * (~mask).mean(), 3)}
    for nama, mask in aturan_jul.items()
])

laporan_jan_rename = laporan[["aturan", "persen_gagal"]].rename(columns={"persen_gagal": "persen_gagal_januari"})
laporan_komparatif = laporan_jan_rename.merge(laporan_jul, on="aturan", how="left")
laporan_komparatif["selisih"] = laporan_komparatif["persen_gagal_juli"] - laporan_komparatif["persen_gagal_januari"]

display(laporan_komparatif.sort_values("selisih", key=abs, ascending=False))

# Eksekusi Pipeline Juli 2023
# Tambahkan cuaca Juli ke df cuaca global agar join K-10 berhasil
cuaca_jul = ambil_cuaca("2023-07-01", "2023-07-31")
cuaca_jul["jam_mulai"] = pd.to_datetime(cuaca_jul["time"])
cuaca_jul = cuaca_jul.rename(columns={"temperature_2m": "suhu_c", "precipitation": "hujan_mm"})
cuaca = pd.concat([cuaca, cuaca_jul], ignore_index=True)

pipeline_inkremental("2023-07", URL_JUL)


[Tugas R.1 & R.2: Pipeline Dua Bulan & Profil Komparatif]
Unduhan selesai: 46.12 MB dalam 0.99 s (46.42 MB/s)


,aturan,persen_gagal_januari,persen_gagal_juli,selisih
0,kelengkapan: passenger_count ada,2.339,2.927,0.588
4,konsistensi: total >= fare,0.816,1.053,0.237
3,validitas: total_amount > 0,0.840,1.075,0.235
1,validitas: jarak 0-100 mil,1.498,1.717,0.219
2,konsistensi: durasi 1-180 menit,1.186,1.311,0.125
6,keunikan: baris tidak duplikat,0.000,0.000,0.000
7,validitas: PULocationID dikenal,0.000,0.000,0.000
5,ketepatan waktu dalam Jan 2023,0.002,NaN,NaN


Memproses bulan 2023-07...
cache ditemukan: yellow_tripdata_2023-07.parquet
Validasi lulus: 2,817,657 baris x 18 kolom
Selesai! 2,817,657 baris ditambahkan ke /content/lapisan_terkurasi/trips_bersih.


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,tip_amount,total_amount,durasi_menit,jam,borough_naik,nama_pembayaran,jam_mulai,suhu_c,hujan_mm,time,tanggal
0,2023-07-01 00:29:59,2023-07-01 00:40:15,1.0,1.80,140,263,1,12.10,5.10,22.20,10.266667,0,Manhattan,Kartu kredit,2023-07-01 00:00:00,19.3,0.0,2023-07-01T00:00,2023-07-01
1,2023-07-01 00:03:25,2023-07-01 00:23:44,1.0,2.31,163,163,2,19.10,0.00,24.10,20.316667,0,Manhattan,Tunai,2023-07-01 00:00:00,19.3,0.0,2023-07-01T00:00,2023-07-01
2,2023-07-01 00:38:29,2023-07-01 00:48:53,1.0,2.36,142,262,1,13.50,3.70,22.20,10.400000,0,Manhattan,Kartu kredit,2023-07-01 00:00:00,19.3,0.0,2023-07-01T00:00,2023-07-01
3,2023-07-01 00:14:16,2023-07-01 00:29:13,1.0,4.36,68,24,1,19.80,4.96,29.76,14.950000,0,Manhattan,Kartu kredit,2023-07-01 00:00:00,19.3,0.0,2023-07-01T00:00,2023-07-01
4,2023-07-01 00:11:15,2023-07-01 00:20:47,0.0,1.60,161,107,1,11.40,3.25,19.65,9.533333,0,Manhattan,Kartu kredit,2023-07-01 00:00:00,19.3,0.0,2023-07-01T00:00,2023-07-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2817652,2023-07-31 23:01:00,2023-07-31 23:19:00,1.0,0.92,142,224,0,22.95,5.39,32.34,18.000000,23,Manhattan,NaN,2023-07-31 23:00:00,19.4,0.0,2023-07-31T23:00,2023-07-31
2817653,2023-07-31 23:55:10,2023-07-31 23:58:40,1.0,1.00,61,188,0,14.44,0.00,15.94,3.500000,23,Brooklyn,NaN,2023-07-31 23:00:00,19.4,0.0,2023-07-31T23:00,2023-07-31
2817654,2023-07-31 23:29:31,2023-07-31 23:51:17,1.0,4.97,238,243,0,22.12,0.00,23.62,21.766667,23,Manhattan,NaN,2023-07-31 23:00:00,19.4,0.0,2023-07-31T23:00,2023-07-31
2817655,2023-07-31 23:29:00,2023-07-31 23:46:00,1.0,3.24,65,62,0,9.54,3.00,14.04,17.000000,23,Brooklyn,NaN,2023-07-31 23:00:00,19.4,0.0,2023-07-31T23:00,2023-07-31


R.3 (Sumber Keempat: API Publik Hari Libur Nasional US)

In [20]:
print("\n[Tugas R.3: API Publik Hari Libur Nasional US]")
URL_HOLIDAY = "https://date.nager.at/api/v3/PublicHolidays/2023/US"

def ambil_libur(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    df_libur = pd.DataFrame(r.json())[["date", "name"]].rename(columns={"name": "nama_libur"})
    df_libur["is_libur"] = 1
    return df_libur

df_libur = ambil_libur(URL_HOLIDAY)
catat_sumber("hari_libur", URL_HOLIDAY, len(df_libur), "Nager.Date US Public Holidays 2023")
display(df_libur.head())


[Tugas R.3: API Publik Hari Libur Nasional US]
[lineage] hari_libur: 17 baris


,date,nama_libur,is_libur
0,2023-01-02,New Year's Day,1
1,2023-01-16,"Martin Luther King, Jr. Day",1
2,2023-02-12,Lincoln's Birthday,1
3,2023-02-20,Presidents Day,1
4,2023-04-07,Good Friday,1


R.4 (Pembuatan kamus_data.md)

In [21]:
print("\n[Tugas R.4: Generate Kamus Data]")
kamus_md = """# Kamus Data: Dataset Analitik Taksi NYC

| Nama Kolom | Tipe Data | Satuan | Sumber Asal | Penanganan Missing Value | Aturan Validitas | Keterangan |
|---|---|---|---|---|---|---|
| `tpep_pickup_datetime` | datetime | Waktu lokal | TLC Parquet | Drop | Sesuai target bulan | Waktu penumpang naik |
| `tpep_dropoff_datetime` | datetime | Waktu lokal | TLC Parquet | Drop | > pickup_datetime | Waktu penumpang turun |
| `passenger_count` | float64 | Orang | TLC Parquet | Isi median per jam | not null | Jumlah penumpang |
| `trip_distance` | float64 | Mil | TLC Parquet | - | 0.01 - 100 mil | Jarak perjalanan |
| `trip_distance_capped` | float64 | Mil | Rekayasa | - | Maksimal P99.5 | Jarak setelah di-winsorize |
| `durasi_menit` | float64 | Menit | Rekayasa | - | 1 - 180 menit | Waktu tempuh |
| `total_amount` | float64 | USD ($) | TLC Parquet | Drop | > 0 | Total tarif |
| `tarif_ekstrem` | int8 | Boolean | Rekayasa | - | 0 atau 1 | Penanda outlier IQR |
| `borough_naik` | string | Kategori | Taxi Zone CSV | Drop | Valid borough | Area penjemputan |
| `payment_type` | int64 | Kategori | TLC Parquet | Drop | 1 s.d. 6 | Kode pembayaran |
| `nama_pembayaran` | string | Kategori | SQLite Latihan| Drop | Ada di tabel dimensi | Label teks pembayaran |
| `suhu_c` | float64 | Celcius | Open-Meteo API| Dibiarkan NaN | - | Suhu saat penjemputan |
| `hujan_mm` | float64 | mm | Open-Meteo API| Dibiarkan NaN | >= 0 | Curah hujan (mm) |
| `hujan` | int8 | Boolean | Rekayasa | 0 (Kering) | 0 atau 1 | 1 jika hujan_mm > 0.1 |
| `tanggal` | string | YYYY-MM-DD| Rekayasa | - | Format ISO valid | Kolom partisi |
| `is_libur` | int8 | Boolean | API Nager.Date| 0 jika tak match | 0 atau 1 | 1 jika libur nasional US |
"""

PATH_KAMUS = os.path.join(DIR_SIMPAN, "kamus_data.md")
with open(PATH_KAMUS, "w") as f:
    f.write(kamus_md)
print(f"Kamus data tersimpan di: {PATH_KAMUS}")


[Tugas R.4: Generate Kamus Data]
Kamus data tersimpan di: /content/drive/MyDrive/BigData/Praktikum03/kamus_data.md
